In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import dwave_networkx as dnx

# B0 = 8.58633500 # coeff in front of ising term, in GHz, at s=1
# B0 = 0.2596733
B0 = 1.0
h = 6.62607015e-34  # Planck constant, in J/Hz
kb = 1.380649e-23  # Boltzmann constant, in J/K
units_factor = (B0/2) * 10**9 * h
beta1 = kb / units_factor  # beta1 = 1
# units_factor = 1
# kb = 1

import matplotlib as mpl
def latex_plot(scale=1, fontsize=12):
    """Changes the size of a figure and fonts for the publication-quality plots."""
    fig_width_pt = 246.0
    inches_per_pt = 1.0 / 72.27
    golden_mean = (np.sqrt(5.0) - 1.0) / 2.0
    fig_width = fig_width_pt * inches_per_pt * scale
    fig_height = fig_width * golden_mean
    fig_size = [fig_width, fig_height]
    eps_with_latex = {
        "pgf.texsystem": "pdflatex",
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": [],
        "font.sans-serif": [],
        "font.monospace": [],
        "axes.labelsize": fontsize,
        "font.size": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "figure.figsize": fig_size,
    }
    mpl.rcParams.update(eps_with_latex)
ROOT = os.getcwd()
if os.path.basename(ROOT) != "plots" and os.path.exists(os.path.join(ROOT, "plots", "plots_2d.ipynb")):
    ROOT = os.path.join(ROOT, "plots")


## Power scaling with annealing parameter

In [ ]:
names = ["RAU"]#, "RAU", "CBFM"]
chain_lengths = [6]
# Remove fixed beta1, will extract all beta1 from data
# Helper to compute nanmean safely
import numpy as np
def _nanmean_list(vals):
    arr = np.array(vals, dtype=float)
    if arr.size == 0:
        return np.nan
    return np.nanmean(arr)
def _extract_params_from_data(data_dict):
    """Extract unique annealing parameters, times, and beta1s from data dictionary keys."""
    annealing_params_set = set()
    annealing_times_set = set()
    beta1_set = set()
    if not isinstance(data_dict, dict):
        return [], [], []
    for k in data_dict.keys():
        if isinstance(k, tuple):
            if len(k) == 4:
                # Format: (run, beta1, at, ap)
                try:
                    beta1_val = float(k[1])
                    at_val = int(k[2]) if isinstance(k[2], (int, str)) else int(float(k[2]))
                    ap_val = float(k[3]) if isinstance(k[3], (float, str)) else float(k[3])
                    beta1_set.add(beta1_val)
                    annealing_times_set.add(at_val)
                    annealing_params_set.add(ap_val)
                except (ValueError, TypeError):
                    continue
            elif len(k) == 3:
                # Legacy format: (run, at, ap)
                try:
                    at_val = int(k[1]) if isinstance(k[1], (int, str)) else int(float(k[1]))
                    ap_val = float(k[2]) if isinstance(k[2], (float, str)) else float(k[2])
                    annealing_times_set.add(at_val)
                    annealing_params_set.add(ap_val)
                except (ValueError, TypeError):
                    continue
            elif len(k) == 2:
                # Legacy single-run: (at, ap)
                try:
                    at_val = int(k[0]) if isinstance(k[0], (int, str)) else int(float(k[0]))
                    ap_val = float(k[1]) if isinstance(k[1], (float, str)) else float(k[1])
                    annealing_times_set.add(at_val)
                    annealing_params_set.add(ap_val)
                except (ValueError, TypeError):
                    continue
    return sorted(list(annealing_params_set)), sorted(list(annealing_times_set)), sorted(list(beta1_set))
# Process and aggregate data across runs into `results`
results = {}
for chain_length in chain_lengths:
    num_var = dnx.pegasus_graph(chain_length).number_of_nodes()
    print(f"Number of variables: {num_var}")
    all_annealing_params = set()
    all_annealing_times = set()
    all_beta1s = set()
    for name in names:
        beta_data = np.load(
            f"{ROOT}/../data/results/phase_diagram_P{chain_length}_{name}_advantage6.4/betas2_P{chain_length}.pkl",
            allow_pickle=True,
        )
        Q_data = np.load(
            f"{ROOT}/../data/results/phase_diagram_P{chain_length}_{name}_advantage6.4/Q_P{chain_length}.pkl",
            allow_pickle=True,
        )
        if hasattr(beta_data, "item"):
            beta_data = beta_data.item()
        if hasattr(Q_data, "item"):
            Q_data = Q_data.item()
        params, times, beta1s = _extract_params_from_data(beta_data)
        all_annealing_params.update(params)
        all_annealing_times.update(times)
        all_beta1s.update(beta1s)
        params_q, times_q, beta1s_q = _extract_params_from_data(Q_data)
        all_annealing_params.update(params_q)
        all_annealing_times.update(times_q)
        all_beta1s.update(beta1s_q)
    annealing_params = sorted(list(all_annealing_params))
    annealing_times = sorted(list(all_annealing_times))
    beta1s = sorted(list(all_beta1s))
    print(f"Detected annealing parameters: {annealing_params}")
    print(f"Detected annealing times: {annealing_times}")
    print(f"Detected beta1s: {beta1s}")
    # Allocate arrays [name, beta1, anneal_param, anneal_time]
    shape = (len(names), len(beta1s), len(annealing_params), len(annealing_times))
    beta2 = np.full(shape, np.nan)
    work = np.full(shape, np.nan)
    power = np.full(shape, np.nan)
    dE = np.full(shape, np.nan)
    varE = np.full(shape, np.nan)
    gval = np.full(shape, np.nan)
    Tgval = np.full(shape, np.nan)
    betas = np.full(shape, np.nan)
    dE2 = np.full(shape, np.nan)
    g = lambda x: x * np.arctanh(x) * kb
    def _keys_for_at_ap_beta1(data_dict, at, ap, beta1):
        if not isinstance(data_dict, dict) or len(data_dict) == 0:
            return []
        at_key = f"{at:d}"
        ap_key = f"{ap:.3f}"
        beta1_key = f"{beta1:.3f}"
        keys = []
        for k in data_dict.keys():
            if isinstance(k, tuple):
                if len(k) == 4:
                    # (run, beta1, at, ap)
                    if (str(k[2]) == at_key) and (str(k[3]) == ap_key) and (f"{float(k[1]):.3f}" == beta1_key):
                        keys.append(k)
                elif len(k) == 3:
                    if (str(k[1]) == at_key) and (str(k[2]) == ap_key):
                        keys.append(k)
                elif len(k) == 2:
                    if (str(k[0]) == at_key) and (str(k[1]) == ap_key):
                        keys.append(k)
        return keys
    for i, name in enumerate(names):
        print(f"Processing {name}")
        beta_data = np.load(
            f"{ROOT}/../data/results/phase_diagram_P{chain_length}_{name}_advantage6.4/betas2_P{chain_length}.pkl",
            allow_pickle=True,
        )
        Q_data = np.load(
            f"{ROOT}/../data/results/phase_diagram_P{chain_length}_{name}_advantage6.4/Q_P{chain_length}.pkl",
            allow_pickle=True,
        )
        if hasattr(beta_data, "item"):
            beta_data = beta_data.item()
        if hasattr(Q_data, "item"):
            Q_data = Q_data.item()
        for bidx, beta1 in enumerate(beta1s):
            for j, ap in enumerate(annealing_params):
                for k, at in enumerate(annealing_times):
                    keys_b = _keys_for_at_ap_beta1(beta_data, at, ap, beta1)
                    keys_q = _keys_for_at_ap_beta1(Q_data, at, ap, beta1)
                    beta2_vals = [(-beta_data[kbkey] / units_factor) * kb for kbkey in keys_b]
                    Qmeans = [(Q_data[kq][0]) * units_factor for kq in keys_q]
                    Qvars = [(Q_data[kq][1]) * (units_factor**2) for kq in keys_q]
                    beta2_avg = _nanmean_list(beta2_vals)
                    Qmean_avg = _nanmean_list(Qmeans)
                    Qvar_avg = _nanmean_list(Qvars)
                    if np.isnan(beta2_avg) or np.isnan(Qmean_avg) or np.isnan(Qvar_avg):
                        beta2[i, bidx, j, k] = np.nan
                        work[i, bidx, j, k] = np.nan
                        power[i, bidx, j, k] = np.nan
                        dE[i, bidx, j, k] = np.nan
                        varE[i, bidx, j, k] = np.nan
                        gval[i, bidx, j, k] = np.nan
                        Tgval[i, bidx, j, k] = np.nan
                        betas[i, bidx, j, k] = np.nan
                        dE2[i, bidx, j, k] = np.nan
                        continue
                    beta2[i, bidx, j, k] = beta2_avg
                    x = Qmean_avg / np.sqrt(Qvar_avg + Qmean_avg**2)
                    gx = g(x)
                    work[i, bidx, j, k] = 2 / (beta2[i, bidx, j, k]) * gx + (1 - beta1 / beta2[i, bidx, j, k]) * Qmean_avg
                    power[i, bidx, j, k] = work[i, bidx, j, k] / (at * 10 ** (-6))
                    dE[i, bidx, j, k] = Qmean_avg
                    varE[i, bidx, j, k] = Qvar_avg
                    gval[i, bidx, j, k] = gx
                    Tgval[i, bidx, j, k] = gx * 2 / (beta2[i, bidx, j, k])
                    betas[i, bidx, j, k] = (1 - beta1 / beta2[i, bidx, j, k])
                    dE2[i, bidx, j, k] = gx * 2 / (beta2[i, bidx, j, k]) - beta1/beta2[i, bidx, j, k] * Qmean_avg
    results[chain_length] = dict(
        beta2=beta2,
        work=work,
        power=power,
        dE=dE,
        dE2=dE2,
        varE=varE,
        gval=gval,
        Tgval=Tgval,
        betas=betas,
        num_var=num_var,
        annealing_params=annealing_params,
        annealing_times=annealing_times,
        beta1s=beta1s,
    )

In [ ]:
sorted(beta1s_q)

## Plotting

In [ ]:
# Create plots from precomputed `results`
font_size = 12  # Control all font sizes with this parameter
line_style = "-"  # Line style for all plots
line_width = 0.5  # Line width for all plots
marker_size = 2  # Marker size for all plots
markers_list = ["o", "s", "^", "v", "D", "*"]  # List of markers for different annealing times

names = ["CON"] #, "RAU", "CBFM"]
for i, name in enumerate(names):
  for chain_length in chain_lengths:
    arr = results[chain_length]
    num_var = arr["num_var"]
    annealing_params = arr["annealing_params"]
    annealing_times = arr["annealing_times"]
    beta1s = arr["beta1s"]
    print(f"Number of variables: {num_var}")
    print(f"Plotting with {len(annealing_params)} annealing params, {len(annealing_times)} times, {len(beta1s)} beta1s")

    latex_plot(scale=1, fontsize=font_size)
    from matplotlib import gridspec
    fig = plt.figure(figsize=(11, 6))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.15, wspace=0.35)
    axs = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(3)]

    beta2 = arr["beta2"]
    work = arr["work"]
    power = arr["power"]
    dE = arr["dE"]
    varE = arr["varE"]
    gval = arr["gval"]
    dE2 = arr["dE2"]
    Tgval = arr["Tgval"]
    betas = arr["betas"]

    blist = [0.7, 0.3, 0.1, 0.02, 0.06, 0.002]

    # For each annealing time, plot curves for all beta1s
    for k, at in enumerate(annealing_times):
      for bidx, beta1 in enumerate(beta1s):
        if beta1 not in blist:
            continue
        label = fr"$\beta_1={beta1:.3f}$, {at} $\mu s$" if len(beta1s) > 1 else fr"{at} $\\mu s$"
        color = None  # Let matplotlib choose, or use a colormap if desired
        axs[0].plot(
          annealing_params,
          1 / beta2[i, bidx, :, k] * 1000,
          label=label,
          marker=markers_list[k % len(markers_list)],
          markersize=marker_size,
          linestyle=line_style,
          linewidth=line_width,
          color=color,
        )
        print(beta2[i, bidx, :, k])
      axs[0].axhline(5, color="red", linestyle="--", linewidth=0.8)
      axs[0].text(0.2, 6, "5 mK", color="red", fontsize=10)
      axs[0].set_xlabel("Annealing parameter")
      axs[0].set_ylabel(r"Temperature $T_2$ [mK]", fontsize=font_size)
      axs[0].legend(loc='upper left', frameon=False, markerscale=1, fontsize=10)
      axs[0].set_yscale('log')

      # Work
      for bidx, beta1 in enumerate(beta1s):
        if beta1 not in blist:
            continue
        axs[1].plot(
          annealing_params,
          work[i, bidx, :, k],
          label=fr"$\\beta_1={beta1:.3f}$" if len(beta1s) > 1 else None,
          marker=markers_list[k % len(markers_list)],
          markersize=marker_size,
          linestyle=line_style,
          linewidth=line_width,
        )
      axs[1].set_xlabel(r"Annealing parameter $s$")
      axs[1].set_ylabel(r"Work bound $\langle W \rangle/L$ [J]", fontsize=font_size, labelpad=-5)
      axs[1].axhline(0, color="gray", linestyle="--", linewidth=0.5)

      # Power
      for bidx, beta1 in enumerate(beta1s):
        if beta1 not in blist:
            continue
        axs[2].plot(
          annealing_params,
          power[i, bidx, :, k],
          label=fr"$\\beta_1={beta1:.3f}$" if len(beta1s) > 1 else None,
          marker=markers_list[k % len(markers_list)],
          markersize=marker_size,
          linestyle=line_style,
          linewidth=line_width,
        )
      axs[2].set_xlabel(r"Annealing parameter $s$")
      axs[2].set_ylabel(r"Power bound $\langle P \rangle/L$ [W]", fontsize=font_size, labelpad=-2)
      axs[2].axhline(0, color="gray", linestyle="--", linewidth=0.5)

      # dE
      for bidx, beta1 in enumerate(beta1s):
        if beta1 not in blist:
            continue
        axs[3].plot(
          annealing_params,
          dE[i, bidx, :, k],
          label=fr"$\\beta_1={beta1:.3f}$" if len(beta1s) > 1 else None,
          marker=markers_list[k % len(markers_list)],
          markersize=marker_size,
          linestyle=line_style,
          linewidth=line_width,
        )
      axs[3].set_xlabel(r"Annealing parameter $s$")
      axs[3].set_ylabel(r"QPU energy change $\langle \Delta E_1 \rangle/L$ [J]")

      # dE2
      for bidx, beta1 in enumerate(beta1s):
        if beta1 not in blist:
            continue
        axs[4].plot(
          annealing_params,
          -dE2[i, bidx, :, k],
          label=fr"$\\beta_1={beta1:.3f}$" if len(beta1s) > 1 else None,
          marker=markers_list[k % len(markers_list)],
          markersize=marker_size,
          linestyle=line_style,
          linewidth=line_width,
        )
      axs[4].set_xlabel(r"Annealing parameter $s$")
      axs[4].set_ylabel(r"Dumped heat bound $\langle Q \rangle/L$ [J]")
      axs[4].axhline(0, color="gray", linestyle="--", linewidth=0.5)

      # (2/beta2) g(Delta E)
      for bidx, beta1 in enumerate(beta1s):
        if beta1 not in blist:
            continue
        axs[5].plot(
          annealing_params,
          varE[i, bidx, :, k] + dE[i, bidx, :, k]**2,
          label=fr"$\beta_1={beta1:.3f}$" if len(beta1s) > 1 else None,
          marker=markers_list[k % len(markers_list)],
          markersize=marker_size,
          linestyle=line_style,
          linewidth=line_width,
        )
      axs[5].set_xlabel(r"Annealing parameter $s$")
      axs[5].set_ylabel(r"$\langle\Delta E_1^2\rangle/L$ [J$^2$]")

    # Add panel labels
    # labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
    for idx, ax in enumerate(axs):
        ax.grid(True, which='major', linestyle='-', linewidth=0.5)
    #     if idx == 0 or idx == 5:
    #         ax.text(0.6, 0.95, labels[idx], transform=ax.transAxes, fontsize=font_size, verticalalignment='top', fontweight='bold')
    #     else:
    #         ax.text(0.05, 0.95, labels[idx], transform=ax.transAxes, fontsize=font_size, verticalalignment='top', fontweight='bold')
    plt.savefig(f"thermodynamics_quantities_P{chain_length}_{name}.pdf", bbox_inches='tight')
    plt.show()

In [ ]:

# Create beta2 vs annealing parameter plot with beta1 colormap
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Use the names list from the data processing cell
names = ["CON"]  # Names that were actually loaded in the data processing
chain_lengths_to_plot = list(results.keys())  # Get chain lengths from results dict

for i, name in enumerate(names):
    for chain_length in chain_lengths_to_plot:
        arr = results[chain_length]
        annealing_params = arr["annealing_params"]
        beta1s = arr["beta1s"]
        beta2 = arr["beta2"]
        
        # Create figure
        latex_plot(scale=1, fontsize=12)
        fig, ax = plt.subplots(figsize=(8, 5))
        
        # Set up colormap
        norm = Normalize(vmin=0, vmax=1)
        cmap = plt.get_cmap('viridis')
        
        # Plot beta2 vs annealing parameter for each beta1
        for bidx, beta1 in enumerate(beta1s):
            color = cmap(norm(beta1))
            # Average over annealing times for cleaner plot
            beta2_mean = np.nanmean(beta2[i, bidx, :, :], axis=1)
            ax.plot(
                annealing_params,
                beta2_mean,
                color=color,
                marker='o',
                markersize=4,
                linestyle='-',
                linewidth=0.75,
                label=f'$\\beta_1={beta1:.3f}$'
            )
        
        # Add colorbar
        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax)
        cbar.set_label(r'$\beta_1$', fontsize=12)
        
        ax.set_xlabel(r'Annealing parameter $s$', fontsize=12)
        ax.set_ylabel(r'$\beta_2$ (unitless)', fontsize=12)
        ax.set_title(f'$\\beta_2$ vs Annealing Parameter - P{chain_length} {name}', fontsize=12)
        ax.grid(True, which='major', linestyle='-', linewidth=0.5, alpha=0.3)
        # ax.legend(loc='best', frameon=False, fontsize=9, ncol=3, bbox_to_anchor=(1.15, 1))
        
        plt.tight_layout()
        plt.savefig(f"beta2_vs_annealing_P{chain_length}_{name}.pdf", bbox_inches='tight')
        plt.show()
